# Activity 1: RAGAS Evaluation with Cost Analysis

Use RAGAS to evaluate your open-source Fireworks AI powered RAG app against an OpenAI gpt-4.1-mini powered equivalent. Compare retrieval quality, answer faithfulness, and end-to-end accuracy across both providers.

Additionally, instrument both pipelines with LangSmith to capture token usage and cost per query. Use LangSmith's tracing and cost dashboards to compare the total cost of running each provider at scale. Include your evaluation results, cost breakdown, and analysis in your Loom video.

**My approach:**
- Build **one** shared retriever (Fireworks embeddings + in-memory Qdrant).
- Run the **same** retrieved context through two generators — Fireworks `gpt-oss-20b` and
  OpenAI `gpt-4.1-mini`.
- Since retrieval is identical, any quality/cost difference comes from the LLM.

In [1]:
# Step 1 — env + LangSmith tracing
import os
from datetime import datetime

from dotenv import load_dotenv

load_dotenv(override=True)

# My app reaches Fireworks via an explicit base_url, so OpenAI calls here must go to real
# OpenAI. Drop any stray OPENAI_BASE_URL/OPENAI_API_BASE so they don't get misrouted.
for _k in ("OPENAI_BASE_URL", "OPENAI_API_BASE"):
    if os.environ.pop(_k, None):
        print(f"cleared stray {_k}")

# Trace to a fresh project each run so the step-7 cost pull only sums this run.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = os.environ.get("LANGSMITH_API_KEY", "")
os.environ["LANGSMITH_PROJECT"] = f"ragas-fw-vs-oai-{datetime.now():%Y%m%d-%H%M%S}"

assert os.environ.get("FIREWORKS_API_KEY"), "Missing FIREWORKS_API_KEY"
assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
assert os.environ.get("LANGSMITH_API_KEY"), "Missing LANGSMITH_API_KEY"

# The embedding model is called against Fireworks, so it must be a Fireworks ID.
_emb = os.environ.get("FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b")
assert not _emb.startswith("text-embedding-"), f"{_emb!r} is an OpenAI embedding name; use a Fireworks ID"

print("env OK — Fireworks embed:", _emb, "| project:", os.environ["LANGSMITH_PROJECT"])

env OK — Fireworks embed: accounts/fireworks/models/qwen3-embedding-8b | project: ragas-fw-vs-oai-20260707-130650


In [2]:
# Step 2 — build the shared retriever. I reuse app/rag.py's build_retriever()
# (load -> chunk -> embed with Fireworks -> in-memory Qdrant) so the eval hits the
# exact same retrieval stack as my app.
from app.rag import build_retriever

retriever = build_retriever()

# sanity check that retrieval works
_docs = retriever.invoke("cat vaccination schedule")
print(f"retriever returned {len(_docs)} chunks; first preview:\n{_docs[0].page_content[:200]}")

retriever returned 4 chunks; first preview:
Practitioners can develop individualized vaccination protocols
consisting of core vaccines (rabies virus, feline herpesvirus type 1
[FHV-1], feline calicivirus [FCV], and feline panleukopenia virus
[F


## 3. The two generators

Both generators share the same prompt. The Fireworks one is reused straight from my app
(`build_fireworks_generator`); for the OpenAI side I use a plain `ChatOpenAI("gpt-4.1-mini")`
that talks to OpenAI directly. Each becomes a `prompt | llm | StrOutputParser()` chain.

In [3]:
# Step 3 — two generator chains sharing one prompt
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

from app.rag import build_prompt, build_fireworks_generator

chat_prompt = build_prompt()

# Fireworks generator, reused from my app
fireworks_llm = build_fireworks_generator()
fireworks_chain = chat_prompt | fireworks_llm | StrOutputParser()

# OpenAI generator — plain ChatOpenAI, hits OpenAI directly (no base_url override)
openai_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
openai_chain = chat_prompt | openai_llm | StrOutputParser()

## 4. Pipeline helper

`run_pipeline` runs one question end to end: retrieve context from the shared retriever,
run the chosen chain, and return the answer plus the raw context chunks (I keep the
chunks because RAGAS needs them).

In [4]:
def run_pipeline(question: str, chain) -> dict:
    """Run one question through the shared retriever + the given chain.

    Returns the answer and the raw context chunks (RAGAS needs both). The prompt
    expects `query` and `context`.
    """
    docs = retriever.invoke(question)
    contexts = [d.page_content for d in docs]
    answer = chain.invoke({"query": question, "context": "\n\n".join(contexts)})
    return {"question": question, "answer": answer, "contexts": contexts}


# quick check on one question
q = "How often should an adult cat be vaccinated?"
print("FIREWORKS:", run_pipeline(q, fireworks_chain)["answer"][:300])
print("\nOPENAI   :", run_pipeline(q, openai_chain)["answer"][:300])

FIREWORKS: I don't know.



OPENAI   : Adult cats should be vaccinated with core vaccines as recommended based on their individual risk exposure. The Task Force supports vaccinating every animal with core vaccines and giving non-core vaccines no more frequently than necessary based on risk. Specifically, revaccination against feline panl


## 5. Evaluation dataset

A small set of cat-health questions, each with a short reference answer I treat as the
ground truth. RAGAS uses the reference for `context_recall` and `context_precision`.

In [5]:
# My eval set: each question has a short reference answer I treat as ground truth.
eval_dataset = [
    {
        "question": "How often should an adult cat be vaccinated?",
        "reference": (
            "Core vaccines for adult cats are typically boosted every 1 to 3 years "
            "depending on the vaccine and the cat's lifestyle and risk, following the "
            "initial kitten series and a booster at about one year of age."
        ),
    },
    {
        "question": "What are common signs that a cat may be sick?",
        "reference": (
            "Warning signs include changes in appetite or thirst, weight loss, "
            "lethargy, hiding, vomiting or diarrhea, changes in litter box habits, "
            "and changes in grooming or behavior."
        ),
    },
    {
        "question": "How often should I feed an adult cat?",
        "reference": (
            "Most adult cats do well on two measured meals a day, with the total "
            "amount based on the cat's weight and body condition to avoid obesity."
        ),
    },
    {
        "question": "Why is parasite control important for cats?",
        "reference": (
            "Regular parasite control protects against fleas, ticks, and intestinal "
            "worms that can cause disease in cats and, in some cases, be transmitted "
            "to people; year-round preventives are commonly recommended."
        ),
    },
    {
        "question": "At what age is a cat considered a senior?",
        "reference": (
            "Cats are generally considered senior around 10 to 11 years of age, and "
            "may need more frequent veterinary check-ups as they get older."
        ),
    },
]

print(f"{len(eval_dataset)} eval questions ready.")

5 eval questions ready.


## 6. Run RAGAS on both providers

I run each question through both chains and score four metrics: `faithfulness`,
`answer_relevancy`, `context_precision`, `context_recall`. RAGAS grades with its own judge
LLM (`gpt-5.4-mini`) plus OpenAI embeddings. Since retrieval is shared, the two context_*
scores land equal — only `faithfulness` and `answer_relevancy` move with the generator.

In [6]:
# ragas 0.4.3 imports langchain_community.chat_models.vertexai, which no longer exists in
# langchain-community 0.4.x. I don't use Vertex, so I register a stub before importing ragas.
import sys
import types

_vx = "langchain_community.chat_models.vertexai"
if _vx not in sys.modules:
    try:
        __import__(_vx)
    except ModuleNotFoundError:
        _stub = types.ModuleType(_vx)

        class ChatVertexAI:  # placeholder, never used
            ...

        _stub.ChatVertexAI = ChatVertexAI
        sys.modules[_vx] = _stub

from langchain_openai import OpenAIEmbeddings
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

# RAGAS is LLM-as-judge, so it needs its own grader. I use gpt-5.4-mini — a neutral model
# that is neither of the two I'm comparing, so no model grades its own answers.
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
print("ragas ready with metrics:", [m.name for m in metrics])

ragas ready with metrics: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']


/var/folders/fw/kwy0s5tj0l39fzpsqntd04nm0000gn/T/ipykernel_76337/96860681.py:23: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/fw/kwy0s5tj0l39fzpsqntd04nm0000gn/T/ipykernel_76337/96860681.py:23: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/var/folders/fw/kwy0s5tj0l39fzpsqntd04nm0000gn/T/ipykernel_76337/96860681.py:23: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (


In [7]:
def build_ragas_dataset(chain, provider_tag: str) -> EvaluationDataset:
    """Run every eval question through `chain` and shape the rows RAGAS wants.

    I tag each run per provider so I can split their token/cost in step 7.
    """
    tagged = chain.with_config({"tags": [provider_tag], "run_name": f"rag-{provider_tag}"})
    rows = []
    for item in eval_dataset:
        out = run_pipeline(item["question"], tagged)
        rows.append(
            {
                "user_input": out["question"],
                "retrieved_contexts": out["contexts"],
                "response": out["answer"],
                "reference": item["reference"],
            }
        )
    return EvaluationDataset.from_list(rows)


fw_eval = build_ragas_dataset(fireworks_chain, "provider:fireworks")
oai_eval = build_ragas_dataset(openai_chain, "provider:openai")
print(f"built ragas datasets from {len(eval_dataset)} questions per provider")

built ragas datasets from 5 questions per provider


In [8]:
# Score both providers with the same judge. This makes several judge calls per row.
fw_result = evaluate(dataset=fw_eval, metrics=metrics, llm=ragas_llm, embeddings=ragas_embeddings)
oai_result = evaluate(dataset=oai_eval, metrics=metrics, llm=ragas_llm, embeddings=ragas_embeddings)

fw_scores = fw_result.to_pandas()
oai_scores = oai_result.to_pandas()

print("=== Fireworks (gpt-oss-20b) ===")
print(fw_result)
print("\n=== OpenAI (gpt-4.1-mini) ===")
print(oai_result)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Exception raised in Job[0]: OutputParserException(Invalid json output: {"statements":[{"statement":"The provided context does not specify an exact vaccination frequency for adult cats.","reason":"The context discusses individualized protocols and says non-core vaccines should be given no more frequently than necessary based on risk, but it does not give a specific exact frequency for adult cats.","verdict":1},{"statement":"Practitioners can develop individualized vaccination protocols based on a cat's life stage, lifestyle, and risk factors.","reason":"The context explicitly says practitioners can develop individualized vaccination protocols based on the patient’s life stage, lifestyle, place of origin, and environmental and epidemiologic factors.","verdict":1},{"statement":"Core vaccines should be given to every cat.","reason":"The context states that veterinarians should vaccinate every animal with core vaccines.","verdict":1},{"statement":"Non-core vaccines should be given only as n

=== Fireworks (gpt-oss-20b) ===
{'faithfulness': 0.6722, 'answer_relevancy': 0.7466, 'context_precision': 1.0000, 'context_recall': 0.4000}

=== OpenAI (gpt-4.1-mini) ===
{'faithfulness': 0.8869, 'answer_relevancy': 0.5971, 'context_precision': 1.0000, 'context_recall': 0.2000}


In [9]:
import pandas as pd

# mean of each metric, side by side
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
metric_cols = [c for c in metric_cols if c in fw_scores.columns]

comparison = pd.DataFrame(
    {
        "fireworks (gpt-oss-20b)": fw_scores[metric_cols].mean(),
        "openai (gpt-4.1-mini)": oai_scores[metric_cols].mean(),
    }
)
comparison["delta (oai - fw)"] = (
    comparison["openai (gpt-4.1-mini)"] - comparison["fireworks (gpt-oss-20b)"]
)
comparison.round(3)

,fireworks (gpt-oss-20b),openai (gpt-4.1-mini),delta (oai - fw)
faithfulness,0.672,0.887,0.215
answer_relevancy,0.747,0.597,-0.150
context_precision,1.000,1.000,0.000
context_recall,0.400,0.200,-0.200


## 7. Cost analysis (LangSmith)

Each generator call is tagged by provider, so I pull the tagged LLM runs back from
LangSmith and sum tokens + cost. LangSmith prices OpenAI automatically but not Fireworks,
so I also compute `est_cost_usd` from a small price table for an apples-to-apples number.
(The RAGAS judge calls are untagged, so they're excluded — this is the cost of producing
the answer, not of running the eval.)

In [10]:
import time

from langsmith import Client

time.sleep(8)  # let LangSmith finish ingesting the traces before I query

client = Client()
project = os.environ["LANGSMITH_PROJECT"]


def usage_for_tag(tag: str) -> dict:
    """Sum tokens + LangSmith cost over all LLM runs carrying `tag`."""
    totals = {
        "runs": 0,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
        "langsmith_cost_usd": 0.0,
    }
    runs = client.list_runs(
        project_name=project, run_type="llm", filter=f'has(tags, "{tag}")'
    )
    for r in runs:
        totals["runs"] += 1
        totals["prompt_tokens"] += r.prompt_tokens or 0
        totals["completion_tokens"] += r.completion_tokens or 0
        totals["total_tokens"] += r.total_tokens or 0
        totals["langsmith_cost_usd"] += float(r.total_cost or 0)
    return totals


fw_usage = usage_for_tag("provider:fireworks")
oai_usage = usage_for_tag("provider:openai")
print("fireworks:", fw_usage)
print("openai   :", oai_usage)

fireworks: {'runs': 5, 'prompt_tokens': 13980, 'completion_tokens': 1778, 'total_tokens': 15758, 'langsmith_cost_usd': 0.0}
openai   : {'runs': 5, 'prompt_tokens': 13655, 'completion_tokens': 687, 'total_tokens': 14342, 'langsmith_cost_usd': 0.0040268}


In [11]:
# Price table in USD per 1M tokens.
# Fireworks gpt-oss-20b (serverless "Standard" serving path), from fireworks.ai:
#   uncached input $0.07/M, cached input $0.04/M, output $0.30/M.
# We use the UNCACHED input rate — RAG prompts vary per query, so they don't hit the cache.
PRICES = {
    "fireworks": {"in": 0.07, "out": 0.30},  # gpt-oss-20b serverless (uncached input)
    "openai": {"in": 0.40, "out": 1.60},     # gpt-4.1-mini
}


def est_cost(usage: dict, key: str) -> float:
    p = PRICES[key]
    return (usage["prompt_tokens"] * p["in"] + usage["completion_tokens"] * p["out"]) / 1_000_000


def summarize(usage: dict, key: str) -> dict:
    return {
        "runs": usage["runs"],
        "prompt_tokens": usage["prompt_tokens"],
        "completion_tokens": usage["completion_tokens"],
        "total_tokens": usage["total_tokens"],
        "langsmith_cost_usd": round(usage["langsmith_cost_usd"], 6),
        "est_cost_usd": round(est_cost(usage, key), 6),
    }


cost_table = pd.DataFrame(
    {
        "fireworks (gpt-oss-20b)": summarize(fw_usage, "fireworks"),
        "openai (gpt-4.1-mini)": summarize(oai_usage, "openai"),
    }
)
cost_table

,fireworks (gpt-oss-20b),openai (gpt-4.1-mini)
runs,5.000000,5.000000
prompt_tokens,13980.000000,13655.000000
completion_tokens,1778.000000,687.000000
total_tokens,15758.000000,14342.000000
langsmith_cost_usd,0.000000,0.004027
est_cost_usd,0.001512,0.006561


In [12]:
# Quality + cost in one table (uses est_cost_usd since LangSmith doesn't price Fireworks).
summary = comparison[["fireworks (gpt-oss-20b)", "openai (gpt-4.1-mini)"]].copy()
summary.loc["est_cost_usd"] = [
    round(est_cost(fw_usage, "fireworks"), 6),
    round(est_cost(oai_usage, "openai"), 6),
]
summary.loc["total_tokens"] = [fw_usage["total_tokens"], oai_usage["total_tokens"]]
print("Quality (RAGAS) + cost, side by side:")
summary.round(4)

Quality (RAGAS) + cost, side by side:


,fireworks (gpt-oss-20b),openai (gpt-4.1-mini)
faithfulness,0.6722,0.8869
answer_relevancy,0.7466,0.5971
context_precision,1.0000,1.0000
context_recall,0.4000,0.2000
est_cost_usd,0.0015,0.0066
total_tokens,15758.0000,14342.0000


## Takeaways

For this experiment, I kept the RAG pipeline the same between both runs. Both Fireworks gpt-oss-20b and OpenAI gpt-4.1-mini used the same retriever (Fireworks qwen3-embedding-8b + Qdrant), same retrieved contexts, and were evaluated with the same judge (gpt-5.4-mini). The goal was to isolate the impact of switching the generator model.

| Metric | Fireworks `gpt-oss-20b` | OpenAI `gpt-4.1-mini` | Notes |
|---|---|---|---|
| **faithfulness** | 0.672 | **0.887** | OpenAI followed the retrieved context more closely |
| **answer_relevancy** | **0.747** | 0.597 | Fireworks generated answers that better matched the questions |
| context_precision | 1.000 | 1.000 | Same result as expected since retrieval was shared |
| context_recall | 0.400 | 0.200 | Expected to match; difference is likely judge variance |
| **est_cost_usd** | **$0.0015** | $0.0066 | OpenAI ≈ **4× more expensive** |
| total_tokens | 15,758 | 14,342 | Fireworks generated slightly more tokens but was still cheaper |

### What I learned:

The comparison was not simply one model winning over the other. OpenAI produced more grounded answers with higher faithfulness, while Fireworks performed better on answer relevancy. Since both models received the same retrieved context, these differences came from how each model interpreted and generated answers from the evidence.

The biggest advantage of Fireworks was cost. It achieved competitive quality at around 1/4 of the price, making it attractive for high-volume use cases where the slight faithfulness tradeoff is acceptable.

The bigger issue I noticed was retrieval quality. Since context_recall stayed relatively low for both models, improving the retriever through better chunking, retrieval parameters, or embeddings would likely improve both pipelines more than just swapping the generator.

### However,

This evaluation only used 5 questions, so the results should be treated as directional. LLM-based judges can vary between runs, which explains why retrieval metrics like context_recall were slightly different even though both models used the same retrieved context.

LangSmith tracked token usage for both runs, but Fireworks pricing required manual calculation using provider pricing because the default cost tracking is OpenAI-focused.